# Smoothed short-term trend

In [ ]:
import numpy as np
import pandas as pd

def synthetic_prices():
    rng = np.random.default_rng(7)
    returns = rng.normal(0.0005, 0.012, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))


In [ ]:
def signal(prices):
    """Smooth daily returns over a 9-day window, then lag one bar."""
    smooth = prices.pct_change().rolling(9, center=True).mean()
    return (smooth.shift(1) * 100).clip(-1, 1).fillna(0.0)


In [ ]:
def backtest(prices, cost_bps=10):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    gross = (positions * asset_returns).mean(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(0.0)
    return gross - turnover * cost_bps / 10_000

def sharpe(pnl):
    pnl_std = pnl.std()
    return float(pnl.mean() / pnl_std * np.sqrt(252)) if pnl_std > 0 else 0.0

print(round(sharpe(backtest(synthetic_prices())), 3))
